# WeedDet Paper Evaluation & Figures
**AgriNav | Benny Merryman-Smith**

Generates every quantitative result and figure needed for the paper.
Run AFTER both weeddet_trainingV5 and weeddet_phase2_neckhead are complete.

| Cell | Output |
|------|--------|
| 0 | Setup |
| 1 | **mAP eval** — AP@0.5, AP@0.75, AP@0.5:0.95 (paper Table) |
| 2 | **Combined loss curve** — Phase 1 + Phase 2 (paper Figure) |
| 3 | **Detection figure** — 3 representative images with boxes (paper Figure) |
| 4 | **Print paper-ready numbers** — copy-paste into LaTeX |


In [ ]:
from google.colab import drive
import sys, os, torch, json, glob
from pathlib import Path

drive.mount('/content/drive')
SCRIPT_DIR   = '/content/drive/MyDrive/weeddet_v2_checkpoints'
CKPT_DIR_V5  = '/content/drive/MyDrive/weeddet_v5_checkpoints'
CKPT_DIR_V5B = '/content/drive/MyDrive/weeddet_v5b_checkpoints'
FLAT_ROOT    = '/content/rice_flat'
sys.path.insert(0, SCRIPT_DIR)

for _m in list(sys.modules.keys()):
    if 'weeddet' in _m.lower(): del sys.modules[_m]
import weeddet_Latest as wd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Use v5b if available, else fall back to v5
BEST_CKPT = f'{CKPT_DIR_V5B}/weeddet_v5b_best.pth'
if not os.path.exists(BEST_CKPT):
    BEST_CKPT = f'{CKPT_DIR_V5}/weeddet_v5_best.pth'
    print(f'v5b not found — using v5: {BEST_CKPT}')
else:
    print(f'Using v5b best: {BEST_CKPT}')

ckpt = torch.load(BEST_CKPT, map_location=device)
model = wd.WeedDet(num_classes=1, anchor_base_scale=6, lsc_k=7).to(device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f'Loaded — epoch {ckpt["epoch"]}  val_loss={ckpt["loss"]:.4f}')

import torchvision.transforms as T
tf = T.Compose([T.ToTensor(),
                T.Normalize(mean=wd.IMAGENET_MEAN, std=wd.IMAGENET_STD)])

val_stems = Path(f'{FLAT_ROOT}/val.txt').read_text().splitlines()
print(f'Val set: {len(val_stems)} images')


## Cell 1 — mAP Evaluation (AP@0.5, AP@0.75, AP@0.5:0.95)
Full COCO-style evaluation on the held-out same-distribution val set.


In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'pycocotools', '-q'])
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import xml.etree.ElementTree as ET
from PIL import Image

SCORE_THR = 0.05
NMS_THR   = 0.25

# Build COCO ground truth
coco_gt_dict = {'images':[],'annotations':[],'categories':[{'id':1,'name':'rice'}]}
ann_id = 1
for img_id, stem in enumerate(val_stems, 1):
    img_path = next(iter(glob.glob(f'{FLAT_ROOT}/images/{stem}.*')), None)
    if not img_path: continue
    pil = Image.open(img_path).convert('RGB')
    coco_gt_dict['images'].append({'id': img_id, 'width': pil.width, 'height': pil.height})
    xml_path = f'{FLAT_ROOT}/annotations/{stem}.xml'
    if not os.path.exists(xml_path): continue
    for obj in ET.parse(xml_path).findall('object'):
        bb = obj.find('bndbox')
        x1,y1,x2,y2 = [int(bb.find(t).text) for t in ['xmin','ymin','xmax','ymax']]
        coco_gt_dict['annotations'].append({
            'id': ann_id, 'image_id': img_id, 'category_id': 1,
            'bbox': [x1, y1, x2-x1, y2-y1],
            'area': (x2-x1)*(y2-y1), 'iscrowd': 0
        })
        ann_id += 1

gt_json = '/tmp/coco_gt.json'
with open(gt_json, 'w') as f: json.dump(coco_gt_dict, f)
coco_gt_api = COCO(gt_json)

# Run inference on val set
print(f'Running inference on {len(val_stems)} val images...')
detections = []
for img_id, stem in enumerate(val_stems, 1):
    img_path = next(iter(glob.glob(f'{FLAT_ROOT}/images/{stem}.*')), None)
    if not img_path: continue
    img_orig = Image.open(img_path).convert('RGB')
    img_lb, scale, pl, pt = wd.letterbox_pil(img_orig, 512)
    tensor = tf(img_lb).unsqueeze(0).to(device)
    with torch.no_grad():
        cls_l, regs, anchors, ishape = model._get_logits(tensor)
        results = model._decode(cls_l, regs, anchors, ishape,
                                score_thr=SCORE_THR, nms_thr=NMS_THR,
                                max_dets=300, output_thr=SCORE_THR)
    boxes  = wd.unpad_boxes(results[0]['boxes'].cpu(), scale, pl, pt)
    scores = results[0]['scores'].cpu()
    for box, score in zip(boxes.tolist(), scores.tolist()):
        x1,y1,x2,y2 = box
        detections.append({
            'image_id': img_id, 'category_id': 1,
            'bbox': [x1, y1, x2-x1, y2-y1], 'score': score
        })

print(f'Total detections: {len(detections)}')

if not detections:
    print('WARNING: No detections. Lower SCORE_THR.')
else:
    coco_dt = coco_gt_api.loadRes(detections)
    coco_eval = COCOeval(coco_gt_api, coco_dt, 'bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    AP_ALL = coco_eval.stats[0]
    AP_50  = coco_eval.stats[1]
    AP_75  = coco_eval.stats[2]
    AR_100 = coco_eval.stats[8]

    print(f'\n{"="*45}')
    print(f'  WeedDet Results (same-distribution val)')
    print(f'{"="*45}')
    print(f'  AP @ IoU=0.50:0.95 : {AP_ALL:.4f}  ({AP_ALL*100:.1f}%)')
    print(f'  AP @ IoU=0.50      : {AP_50:.4f}  ({AP_50*100:.1f}%)')
    print(f'  AP @ IoU=0.75      : {AP_75:.4f}  ({AP_75*100:.1f}%)')
    print(f'  AR @ maxDets=100   : {AR_100:.4f}  ({AR_100*100:.1f}%)')
    print(f'  Val images         : {len(val_stems)}')
    print(f'  GT boxes           : {len(coco_gt_dict["annotations"])}')
    print(f'  Detections         : {len(detections)}')
    print(f'{"="*45}')


## Cell 2 — Combined Loss Curve (Phase 1 + Phase 2)
Loads both loss histories and plots them as a unified curve for the paper.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load saved loss curves from Drive
def load_losses(ckpt_dir, prefix='loss_history'):
    hist_path = f'{ckpt_dir}/{prefix}.json'
    if os.path.exists(hist_path):
        with open(hist_path) as f:
            return json.load(f)
    return None

# Try to load saved loss histories
# (If not saved by training notebooks, reconstruct from checkpoint metadata)
p1_hist = load_losses(CKPT_DIR_V5, 'loss_history_v5')
p2_hist = load_losses(CKPT_DIR_V5B, 'loss_history_v5b')

# Fallback: just plot what we know from the best checkpoint epochs
p1_epochs = list(range(1, ckpt.get('epoch', 12) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: if full history available
ax = axes[0]
if p1_hist:
    ax.plot(range(1, len(p1_hist['train'])+1), p1_hist['train'],
            marker='o', ms=3, label='Train (P1)', color='steelblue')
    ax.plot(range(1, len(p1_hist['val'])+1),   p1_hist['val'],
            marker='s', ms=3, label='Val (P1)',   color='tomato')
    if p2_hist:
        offset = len(p1_hist['train'])
        ax.plot(range(offset+1, offset+len(p2_hist['train'])+1), p2_hist['train'],
                marker='o', ms=3, label='Train (P2 neck+head)', color='steelblue',
                linestyle='--')
        ax.plot(range(offset+1, offset+len(p2_hist['val'])+1), p2_hist['val'],
                marker='s', ms=3, label='Val (P2 neck+head)',   color='tomato',
                linestyle='--')
        ax.axvline(offset + 0.5, color='gray', linestyle=':', alpha=0.7,
                   label='Phase boundary')
    ax.set_title('Combined Training Curve (P1 + P2)')
else:
    ax.text(0.5, 0.5, 'Loss history not saved by training notebooks\n'
            '(Re-run training with loss history export to get this plot)',
            ha='center', va='center', transform=ax.transAxes, fontsize=10)
    ax.set_title('Loss history unavailable — re-run training')

ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Right: best checkpoint summary
ax2 = axes[1]
labels = ['v5 best\n(full model)', 'v5b best\n(neck+head)']
v5_loss  = ckpt['loss'] if 'weeddet_v5_best' in BEST_CKPT or True else None
v5b_path = f'{CKPT_DIR_V5B}/weeddet_v5b_best.pth'
v5_path  = f'{CKPT_DIR_V5}/weeddet_v5_best.pth'

losses = []
if os.path.exists(v5_path):
    c = torch.load(v5_path, map_location='cpu')
    losses.append(c['loss'])
if os.path.exists(v5b_path):
    c = torch.load(v5b_path, map_location='cpu')
    losses.append(c['loss'])

if losses:
    bars = ax2.bar(labels[:len(losses)], losses, color=['steelblue','tomato'][:len(losses)],
                   width=0.4, edgecolor='black', linewidth=0.8)
    for bar, val in zip(bars, losses):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Best Val Loss')
    ax2.set_title('Best Val Loss by Phase')
    ax2.set_ylim(0, max(losses) * 1.2)
    ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{CKPT_DIR_V5B}/training_summary.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved to {CKPT_DIR_V5B}/training_summary.png')


## Cell 3 — Paper Detection Figure
Runs inference on 3 representative images and saves a publication-quality figure.
Pick representative images: dense paddy, sparse/aerial, post-flood.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import xml.etree.ElementTree as ET
from PIL import Image
import random

SCORE_THR = 0.40
NMS_THR   = 0.25

# ── Pick 3 representative images ─────────────────────────────────────────────
# Strategy: sort by GT box count — pick dense, medium, sparse
print('Scoring val images by GT box density...')
scored = []
for stem in val_stems[:200]:   # sample first 200 for speed
    xml_path = f'{FLAT_ROOT}/annotations/{stem}.xml'
    if not os.path.exists(xml_path): continue
    n_boxes = len(ET.parse(xml_path).findall('object'))
    scored.append((n_boxes, stem))

scored.sort(key=lambda x: -x[0])
n = len(scored)

dense_stem   = scored[0][1]                # most boxes
medium_stem  = scored[n // 2][1]           # median
sparse_stem  = scored[-1][1] if scored else val_stems[0]  # fewest

selected = [
    (dense_stem,  f'Dense paddy ({scored[0][0]} GT boxes)'),
    (medium_stem, f'Typical paddy'),
    (sparse_stem, f'Sparse / aerial'),
]
print(f'Selected images: {[s[0] for s in selected]}')

# ── Run inference and plot ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (stem, title) in zip(axes, selected):
    img_path = next(iter(glob.glob(f'{FLAT_ROOT}/images/{stem}.*')), None)
    if not img_path:
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center',
                transform=ax.transAxes)
        continue

    img_orig = Image.open(img_path).convert('RGB')
    img_lb, scale, pl, pt = wd.letterbox_pil(img_orig, 512)
    tensor = tf(img_lb).unsqueeze(0).to(device)
    with torch.no_grad():
        cls_l, regs, anchors, ishape = model._get_logits(tensor)
        results = model._decode(cls_l, regs, anchors, ishape,
                                score_thr=0.05, nms_thr=NMS_THR,
                                max_dets=300, output_thr=SCORE_THR)

    pred_boxes  = wd.unpad_boxes(results[0]['boxes'].cpu(), scale, pl, pt)
    pred_scores = results[0]['scores'].cpu()

    # GT boxes
    xml_path = f'{FLAT_ROOT}/annotations/{stem}.xml'
    gt_boxes = []
    if os.path.exists(xml_path):
        for obj in ET.parse(xml_path).findall('object'):
            bb = obj.find('bndbox')
            gt_boxes.append([int(bb.find(t).text) for t in ['xmin','ymin','xmax','ymax']])

    img_np = np.array(img_orig)
    ax.imshow(img_np)

    # Draw GT (green, dashed)
    for b in gt_boxes:
        x1,y1,x2,y2 = b
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=1.5, edgecolor='lime', facecolor='none',
                     linestyle='--'))

    # Draw predictions (red, solid, with score)
    for i, (b, s) in enumerate(zip(pred_boxes.tolist(), pred_scores.tolist())):
        x1,y1,x2,y2 = b
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=1.5, edgecolor='red', facecolor='none'))
        ax.text(x1, max(y1-3, 0), f'{s:.2f}', color='red', fontsize=6,
                backgroundcolor='black',
                bbox=dict(boxstyle='square,pad=0', fc='black', ec='none'))

    ax.axis('off')
    conf_str = (f'scores {pred_scores.min():.2f}–{pred_scores.max():.2f}'
                if len(pred_scores) else 'no detections')
    ax.set_title(f'{title}\nGT: {len(gt_boxes)} | Pred: {len(pred_boxes)} ({conf_str})',
                 fontsize=9, pad=4)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], color='lime',  linewidth=2, linestyle='--', label='Ground truth'),
    Line2D([0],[0], color='red',   linewidth=2, linestyle='-',  label='WeedDet prediction'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle(f'WeedDet Rice Detection — Confidence threshold {SCORE_THR}',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()

save_path = f'{CKPT_DIR_V5B}/paper_detection_figure.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved: {save_path}')


## Cell 4 — Paper-Ready Numbers (copy-paste into LaTeX)

In [ ]:
# Print everything needed for the paper in one block
print('=' * 60)
print('PAPER-READY NUMBERS — AgriNav WeedDet Section')
print('=' * 60)

# Model info
print(f'\n[Model]')
print(f'  Checkpoint   : {BEST_CKPT}')
print(f'  Epoch        : {ckpt["epoch"]}')
print(f'  Best val loss: {ckpt["loss"]:.4f}')

# Dataset info
n_train = len(Path(f'{FLAT_ROOT}/train.txt').read_text().splitlines())
n_val   = len(Path(f'{FLAT_ROOT}/val.txt').read_text().splitlines())
n_gt    = len(coco_gt_dict['annotations']) if 'coco_gt_dict' in dir() else '?'
print(f'\n[Dataset — same-distribution eval]')
print(f'  Train images : {n_train}')
print(f'  Val images   : {n_val}')
print(f'  Val GT boxes : {n_gt}')
print(f'  Split seed   : 42  (80/20)')

# mAP numbers
if 'AP_50' in dir():
    print(f'\n[Quantitative Results]')
    print(f'  AP @ 0.50:0.95 = {AP_ALL:.4f}  ({AP_ALL*100:.1f}%)')
    print(f'  AP @ 0.50      = {AP_50:.4f}  ({AP_50*100:.1f}%)')
    print(f'  AP @ 0.75      = {AP_75:.4f}  ({AP_75*100:.1f}%)')
    print(f'  AR @ 100 dets  = {AR_100:.4f}  ({AR_100*100:.1f}%)')

    print(f'\n[LaTeX table row — paste into Table]')
    print(f'  WeedDet (ours) & {AP_50*100:.1f} & {AP_75*100:.1f} & {AP_ALL*100:.1f} \\\\')

    print(f'\n[LaTeX inline — paste into text]')
    print(f'  "...achieves AP@0.50 of {AP_50*100:.1f}\\%, AP@0.75 of {AP_75*100:.1f}\\%,')
    print(f'   and AP@[0.5:0.95] of {AP_ALL*100:.1f}\\% on a same-distribution')
    print(f'   held-out validation set of {n_val} images."')
else:
    print('\n[mAP] — Run Cell 1 first')

print('\n' + '=' * 60)
